# Deployment Tutorial: From Local to Cloud

> **📓 Notebook · Module 15 — Deployment**
> *Step-by-step guide to deploying Streamlit applications.*

---

## Learning Objectives

After this notebook you will be able to:

1. Prepare a Streamlit app for deployment
2. Organize repository structure correctly
3. Configure requirements, secrets, and config files
4. Deploy to Streamlit Community Cloud
5. Debug common deployment issues
6. Update and redeploy applications

---

## Prerequisites

- Completed Modules 01–14
- GitHub account
- Streamlit Community Cloud account (free)
- Git basics (clone, commit, push)

## 1. The Deployment Workflow

Every deployment follows this cycle:

```
LOCAL DEVELOPMENT → TEST → GIT COMMIT → PUSH TO GITHUB
       ↑                                        ↓
   UPDATE ← REDEPLOY ← MONITOR ← DEPLOY TO CLOUD
```

### Why Deploy?

| Staying Local | Deploying |
|---|---|
| Only you can see it | Anyone with the link can access it |
| Must run on your machine | Runs in the cloud 24/7 |
| No sharing workflow | Share via URL |
| Manual setup on each machine | Instant access |

### When to Deploy

- Sharing a data dashboard with a team
| Demo of an ML model to a client
- Course project submission
- Building a portfolio piece
- Prototyping before production

## 2. Repository Structure for Deployment

Community Cloud expects a clean, minimal structure:

### Minimal Structure

```
my-app/
├── app.py                 # ← Entry point (required)
├── requirements.txt       # ← Dependencies (required)
├── .gitignore            # ← Ignored files
└── .streamlit/
    └── config.toml       # ← Optional config
```

### What Community Cloud Needs

| File | Required? | Purpose |
|------|-----------|--------|
| `app.py` (or chosen entry point) | ✅ Yes | The file to run |
| `requirements.txt` | ✅ Yes | Python packages to install |
| `.streamlit/config.toml` | Optional | Theme and server config |
| Secrets | Optional | Set via UI, never in repo |

### Common Mistakes

❌ Using `streamlit run app.py` as the entry point (Community Cloud just needs the file path)

❌ Forgetting `requirements.txt` — the build will fail

❌ Including `__pycache__/`, `.venv/`, or large data files in the repo

❌ Using absolute file paths like `/Users/me/data.csv`

In [ ]:
# Check that our deployable app has the right structure
import os

required_files = [
    'apps/deployable_app/app.py',
    'apps/deployable_app/requirements.txt',
]

print("Deployment readiness check:")
print("=" * 50)

all_ok = True
for f in required_files:
    exists = os.path.exists(f)
    status = "✅" if exists else "❌"
    print(f"  {status} {f}")
    if not exists:
        all_ok = False

if os.path.exists('apps/deployable_app/requirements.txt'):
    with open('apps/deployable_app/requirements.txt') as f:
        deps = [line.strip() for line in f if line.strip() and not line.startswith('#')]
    print(f"\n  Dependencies listed: {len(deps)}")
    for dep in deps:
        print(f"    • {dep}")

if all_ok:
    print("\n✅ All required files present!")
else:
    print("\n❌ Missing required files — deployment will fail!")

## 3. The requirements.txt File

This file tells Community Cloud which packages to install.

### Best Practices

```txt
# ✅ GOOD: Minimum version with >=
streamlit>=1.44.0
pandas>=2.0.0
numpy>=1.24.0

# ✅ GOOD: Exact version for reproducibility
streamlit==1.44.0

# ❌ BAD: No version specified
streamlit

# ❌ BAD: Using development dependencies
ipython
jupyter
matplotlib
```

### Key Rules

1. **Only include what the app imports** — don't list `jupyter`, `ipython`, etc.
2. **Pin versions** for reproducibility
3. **Put it at the repo root** or in the same directory as your app
4. **Use `>=` for flexibility** or `==` for exact reproducibility

## 4. Secrets Management

### Local Development

Create `.streamlit/secrets.toml` (this is already in `.gitignore`):

```toml
# .streamlit/secrets.toml
[openai]
api_key = "sk-your-key-here"

[database]
host = "localhost"
password = "my-password"
```

### Community Cloud

1. Go to your app on Community Cloud
2. Click **"⋮"** → **"Settings"**
3. Click **"Secrets"** in the sidebar
4. Paste your TOML content
5. Click **"Save"**

### Accessing Secrets in Code

```python
import streamlit as st

# Direct access (raises error if missing)
api_key = st.secrets["openai"]["api_key"]

# Safe access with default
api_key = st.secrets.get("openai", {}).get("api_key", "")
if not api_key:
    st.error("API key not configured. Please add it in app settings.")
    st.stop()
```

## 5. The .gitignore File

Make sure sensitive and unnecessary files are never committed:

```gitignore
# Python
__pycache__/
*.py[cod]
*.egg-info/

# Virtual environments
.venv/
venv/

# Streamlit secrets (NEVER commit!)
.streamlit/secrets.toml

# Large data files
*.csv
*.parquet
uploads/

# OS files
.DS_Store
Thumbs.db
```

### Verify Your .gitignore

Run this to check if secrets are accidentally tracked:

In [ ]:
# Check if .gitignore exists and contains critical entries
gitignore_path = '.gitignore'

critical_entries = [
    'secrets.toml',
    '__pycache__',
    '.venv',
]

print(".gitignore security check:")
print("=" * 50)

if os.path.exists(gitignore_path):
    with open(gitignore_path) as f:
        content = f.read()
    
    for entry in critical_entries:
        found = entry in content
        status = "✅" if found else "⚠️"
        print(f"  {status} Ignores '{entry}'")
else:
    print("  ❌ No .gitignore found!")
    print("  Create one immediately to prevent committing secrets.")

## 6. The .streamlit/config.toml File

Optional configuration for theme and server settings:

```toml
[theme]
base = "light"
primaryColor = "#FF4B4B"
backgroundColor = "#FFFFFF"
secondaryBackgroundColor = "#F0F2F6"
textColor = "#31333F"
font = "sans serif"

[server]
headless = true          # Required for cloud deployment
enableCORS = false
enableXsrfProtection = true

[browser]
gatherUsageStats = false
```

### Important Settings

| Setting | Value | Why |
|---------|-------|-----|
| `headless` | `true` | Prevents opening browser on server |
| `enableXsrfProtection` | `true` | Security against CSRF attacks |
| `gatherUsageStats` | `false` | Privacy — no telemetry sent |
| `enableCORS` | `false` | Security — restricts cross-origin requests |

## 7. Step-by-Step Deployment

### Step 1: Prepare Your App

Before deploying, verify:

```bash
# Run locally to confirm it works
streamlit run app.py

# Check for missing imports
python -c "import streamlit; import pandas; import numpy; print('All imports OK')"

# Check syntax
python -m py_compile app.py
```

### Step 2: Commit and Push

```bash
# Stage your files
git add app.py requirements.txt README.md .gitignore .streamlit/

# Commit
git commit -m "Add deployable dashboard app"

# Push to GitHub
git push origin main
```

### Step 3: Deploy on Community Cloud

1. Go to **[share.streamlit.io](https://share.streamlit.io)**
2. Sign in with your **GitHub** account
3. Click **"New app"**
4. Fill in:
   - **Repository:** `your-username/your-repo`
   - **Branch:** `main`
   - **Main file path:** `app.py` (or `apps/deployable_app/app.py`)
5. Click **"Advanced settings"** to add secrets if needed
6. Click **"Deploy!"**

### Step 4: Wait for Build

Community Cloud will:
1. Clone your repository
2. Install dependencies from `requirements.txt`
3. Run your app

This takes 1–3 minutes on first deploy.

## 8. Debugging Deployment Issues

### Build Failures

| Symptom | Cause | Fix |
|---------|-------|-----|
| `ModuleNotFoundError` | Missing dependency | Add to `requirements.txt` |
| `SyntaxError` | Python syntax error | Fix locally first |
| `No module named 'streamlit'` | Not in requirements | Add `streamlit` to requirements.txt |
| Build timeout | Slow install | Reduce dependencies, pin versions |

### Runtime Errors

| Symptom | Cause | Fix |
|---------|-------|-----|
| `FileNotFoundError` | Missing file in repo | Commit all required files |
| `KeyError` (secrets) | Secret not configured | Add in Community Cloud settings |
| App won't load | Runtime error in script | Check deployment logs |
| App is slow | No caching | Add `@st.cache_data` |
| White page | JavaScript error | Check browser console |

### How to View Logs

1. Go to your app on Community Cloud
2. Click **"⋮"** → **"Manage app"**
3. Click **"Logs"** tab
4. Read the error messages

### Common Fixes

```bash
# If imports fail, add them to requirements.txt:
echo "plotly>=5.15.0" >> requirements.txt

# If file paths break, use relative paths:
# ❌ df = pd.read_csv('/Users/me/data.csv')
# ✅ df = pd.read_csv('data/sales.csv')

# If secrets fail, use safe access:
# ❌ key = st.secrets.openai.api_key
# ✅ key = st.secrets.get('openai', {}).get('api_key', '')
```

## 9. Updating and Redeploying

### Automatic Redeployment

Community Cloud automatically redeploys when you push to GitHub:

```bash
# Make changes to your app
vim app.py

# Commit and push
git add app.py
git commit -m "Update: add new chart"
git push origin main

# Community Cloud will automatically rebuild and redeploy!
```

### Manual Redeployment

If automatic deploy doesn't trigger:

1. Go to app settings on Community Cloud
2. Click **"Reboot app"**

### Rollback

If your update breaks the app:

```bash
# Revert to the previous commit
git revert HEAD
git push origin main

# Community Cloud will deploy the reverted version
```

## 10. Resource Limits and Considerations

### Community Cloud Limits

| Resource | Free Tier |
|----------|----------|
| Memory | ~1 GB |
| CPU | Basic |
| Storage | 50 GB max |
| Apps per account | Limited |
| Sleep | After inactivity |

### App Sleep

Community Cloud apps go to sleep after inactivity. When a user visits:
1. The app wakes up (30–60 seconds cold start)
2. Dependencies are reloaded
3. Cached data is rebuilt

**Mitigation strategies:**
- Use `@st.cache_data` and `@st.cache_resource` to speed up warm-up
- Keep `requirements.txt` lean — fewer packages = faster startup
- Use `persist="disk"` for critical cached data

### Memory Optimization

```python
# ✅ GOOD: Load only what you need
df = pd.read_csv('data.csv', usecols=['date', 'sales', 'category'])

# ❌ BAD: Load everything into memory
df = pd.read_csv('huge_data.csv')  # 2 GB file into memory!

# ✅ GOOD: Cache expensive computations
@st.cache_data
def process_data():
    return heavy_computation()

# ❌ BAD: Recompute on every rerun
def process_data():
    return heavy_computation()  # Runs every time!
```

## 11. Security Best Practices for Deployment

### Never Commit Secrets

```bash
# Check if secrets are accidentally tracked
git ls-files | grep -i secret
git ls-files | grep -i "\.env"
git ls-files | grep -i password

# If found, remove from tracking (keeps local file)
git rm --cached .streamlit/secrets.toml
git commit -m "Remove secrets from tracking"
```

### Validate User Inputs

```python
# ✅ Always validate before processing
user_input = st.text_input("Enter a number")
if user_input:
    try:
        value = float(user_input)
        if value < 0 or value > 100:
            st.error("Value must be between 0 and 100")
        else:
            process(value)
    except ValueError:
        st.error("Please enter a valid number")
```

### Use Parameterized Queries

```python
# ✅ Safe from SQL injection
cursor.execute("SELECT * FROM users WHERE name = ?", (user_name,))

# ❌ Vulnerable to SQL injection
cursor.execute(f"SELECT * FROM users WHERE name = '{user_name}'")
```

## 12. Deployment Checklist

Use this checklist before every deployment:

### Pre-Deployment

- [ ] App runs locally without errors
- [ ] All imports are in `requirements.txt`
- [ ] No hardcoded file paths (use relative paths)
- [ ] No secrets in source code
- [ ] `.gitignore` includes `secrets.toml`
- [ ] `st.set_page_config()` is the first Streamlit call
- [ ] Error handling for user inputs
- [ ] App handles empty/missing data gracefully

### Repository

- [ ] `requirements.txt` at root or app directory
- [ ] Entry point file exists and runs
- [ ] No large data files in repo (< 100 MB)
- [ ] No `__pycache__/`, `.venv/`, or build artifacts
- [ ] README.md with description

### Secrets

- [ ] All secrets configured in Community Cloud settings
- [ ] TOML syntax is valid
- [ ] Code uses `st.secrets.get()` with defaults

### Post-Deployment

- [ ] App loads successfully
- [ ] All features work as expected
- [ ] No errors in deployment logs
- [ ] Performance is acceptable
- [ ] Share URL with stakeholders

## Exercises

### Exercise 1: Prepare an App for Deployment

Take one of your previous exercises and prepare it for deployment:

1. Create a `requirements.txt` with all necessary packages
2. Add a `.gitignore` file
3. Create a `.streamlit/config.toml` with a custom theme
4. Verify the app runs with `streamlit run your_app.py`

### Exercise 2: Deploy to Community Cloud

1. Create a new GitHub repository
2. Push your prepared app
3. Deploy on Community Cloud
4. Verify it works at the public URL
5. Share the URL with a classmate

### Exercise 3: Debug a Broken Deployment

Create a deployment that intentionally fails, then fix it:

1. Push an app without `requirements.txt`
2. Observe the build failure
3. Add the file and redeploy
4. Repeat with missing imports and file paths

### Challenge: Multi-App Deployment

Deploy two different apps from the same repository:
1. App 1: `apps/app_a/dashboard.py`
2. App 2: `apps/app_b/analysis.py`
3. Deploy each with different entry points on Community Cloud

---

## Key Takeaways

1. **Always test locally** before deploying
2. **Keep requirements.txt minimal** — fewer packages = faster deployment
3. **Never commit secrets** — use Community Cloud settings
4. **Use relative file paths** — absolute paths break in the cloud
5. **Monitor your logs** — deployment issues show up there first
6. **Updates auto-deploy** — just push to GitHub

---

## Further Reading

- [Community Cloud Documentation](https://docs.streamlit.io/deploy/streamlit-community-cloud)
- [Deployment Guide](../readings/deployment_guide.md)
- [Troubleshooting Guide](../docs/deployment_troubleshooting.md)
- [Deployment Checklist](../docs/deployment_checklist.md)

---

## Related Materials

- 📖 Reading: [Deployment Guide](../readings/deployment_guide.md)
- 📚 Troubleshooting: [Deployment Issues](../docs/deployment_troubleshooting.md)
- ✏️ Exercises: [Deployment Exercises](../exercises/deployment_exercises.py)
- 📋 Checklist: [Deployment Checklist](../docs/deployment_checklist.md)
- 🖥️ Example: [Deployable App](../apps/deployable_app/)